In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../extraction/caregiverRaw.csv", sep=";")

df

In [ ]:
def eda_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "count": df.count(),
        "missing": df.isna().sum(),
        "missing_%": df.isna().mean() * 100,
        "unique": df.nunique(dropna=True),
    })

    summary["min"] = df.select_dtypes(include="number").min()
    summary["mean"] = df.select_dtypes(include="number").mean()
    summary["median"] = df.select_dtypes(include="number").median()
    summary["max"] = df.select_dtypes(include="number").max()

    return summary.round(2)

summary = eda_summary(df)
summary.to_csv("summary.csv", sep=";")

In [ ]:
dtypes = summary["dtype"].astype(str).value_counts()

plt.bar(
    dtypes.index,
    dtypes.values,
    edgecolor="black"
)

plt.title("Distribution of Data Types in Features")
plt.xlabel("Data Type")
plt.ylabel("Number of Columns")

plt.yticks(
    np.arange(0, dtypes.max() + 5, 5)
)

plt.tight_layout()
plt.show()

In [ ]:
plt.hist(summary["missing_%"], bins=10, edgecolor='black')
plt.title("Distribution of Missing Values (%) in features")
plt.xlabel("Percentage of Missing Values")
plt.ylabel("Number of Columns")
plt.xticks(np.arange(0, 101, 10))
plt.yticks(np.arange(0, 100, 5))
plt.show()

In [ ]:


plt.hist(summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object")]["unique"], bins=40, edgecolor='black')
plt.title("Distribution of unique values in 'object'-dtype columns")
plt.xlabel("Number of unique values")
plt.ylabel("Number of Features")

plt.show()

plt.hist(summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object") & (summary["unique"] <= 100)]["unique"], bins=10, edgecolor='black')
plt.title("Distribution of unique values in 'object'-dtype columns")
plt.xlabel("Number of unique values")
plt.ylabel("Number of Features")
plt.show()

In [ ]:
summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object") & (summary["unique"] <= 100)]

In [ ]:
df["orgRef"].value_counts()

In [ ]:
df[summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object") & (summary["unique"] <= 25)].index]

In [ ]:
from config import pflege_map

average = sum(pflege_map.values()) / len(pflege_map)

In [ ]:
df["pflegestatus-paket"] = (
    df["pflegestatus-paket"]
    .map(pflege_map)
    .fillna(average)
)

In [ ]:
df["pflegestatus-paket"].unique()

In [ ]:
counts = df["orgRef"].value_counts()

df.loc[
    df["orgRef"].map(counts) < 100,
    "orgRef"
] = "Other"


# 2. Group values containing "Übernahme"
df.loc[
    df["orgRef"].str.contains("Übernahme", case=False, na=False),
    "orgRef"
] = "Takeover"


# 3. Group values containing "ZZ"
df.loc[
    df["orgRef"].str.contains("ZZ", case=False, na=False),
    "orgRef"
] = "Former"

df.loc[
    df["orgRef"] == "Sonstiges",
    "orgRef"
] = "Other"

In [ ]:
df["orgRef"].value_counts()

In [ ]:
columns_to_encode = [
    "genderId",
	"nationality",
	"preferredTransport",
	"pers-dat-sprach-mutt",
	"admin-betr-gsber",
	"orgRef"
]

df = pd.get_dummies(
    df,
    columns=columns_to_encode,
    dtype=int,
	drop_first=True
)

In [ ]:
df

In [ ]:
exceptions = [
    "eingestellt-am",
    "ausgesch-am"
]

colsToDrop = [
    col
    for col in summary[summary["missing_%"] > 10].index
    if col not in exceptions
]

df_dropped = df.drop(columns=colsToDrop)

In [ ]:
exceptions = [
    "ausgesch-am"
]

cols_to_check = df_dropped.columns.difference(exceptions)

df_dropped = df_dropped.dropna(subset=cols_to_check)

In [ ]:
df_dropped

In [ ]:

df_dropped["eingestellt-am"] = pd.to_datetime(
    df_dropped["eingestellt-am"].astype("string"),
    format="%Y%m%d%H%M%S",
    errors="coerce"
)

df_dropped["ausgesch-am"] = pd.to_datetime(
    df_dropped["ausgesch-am"].astype("Int64").astype("string"),
    format="%Y%m%d%H%M%S",
    errors="coerce"
)


In [ ]:
df_y = df_dropped

df_y

In [ ]:
feature_summary = pd.DataFrame({
    "nunique": df_y.nunique(),
    "most_common_pct": df_y.apply(
        lambda col: col.value_counts(normalize=True, dropna=False).iloc[0] * 100
    )
})

print(feature_summary.sort_values("most_common_pct", ascending=False))

plt.hist(feature_summary["most_common_pct"], bins= 40, edgecolor="black")
plt.title("Distribution of Most Common Value Percentage Across Features")
plt.xlabel("Most Common Value (%)")
plt.ylabel("Number of Features")
plt.show()

In [ ]:
df_y["dateOfBirth"] = pd.to_datetime(
    df_y["dateOfBirth"].astype("Int64").astype("string"),
    format="%Y%m%d",
    errors="coerce"
)

df_y["birthYear"] = df_y["dateOfBirth"].dt.year
df_y["birthMonth"] = df_y["dateOfBirth"].dt.month



df_y

In [ ]:
df_y.drop(columns="dateOfBirth", inplace=True)

In [ ]:
df_y

In [ ]:
exceptions = [
    "eingestellt-am",
    "ausgesch-am",
    "aussch-grund"
]

df_final = df_y.drop(columns=feature_summary[feature_summary["most_common_pct"] > 99].index)

In [ ]:
df_final

In [ ]:
corr = df_final.corr(numeric_only=True)

# Hide upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Dynamic figure size
n_features = len(corr.columns)
figsize = max(12, n_features * 0.45)

fig, ax = plt.subplots(figsize=(figsize, figsize))

# Mask upper triangle
corr_masked = np.ma.masked_where(mask, corr.values)

im = ax.imshow(
    corr_masked,
    vmin=-1,
    vmax=1,
    aspect="equal"
)

fig.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(range(n_features))
ax.set_yticks(range(n_features))

ax.set_xticklabels(
    corr.columns,
    rotation=90,
    fontsize=8
)

ax.set_yticklabels(
    corr.columns,
    fontsize=8
)

ax.set_title(
    "Correlation Matrix of Model Features",
    fontsize=16,
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
corr = df_final.corr(numeric_only=True)

threshold = 0.8

strong_corr = corr.where(
    (corr.abs() >= threshold) & (corr.abs() < 1)
)

# Remove features that have no strong correlations
keep = strong_corr.notna().any()

strong_corr = strong_corr.loc[keep, keep]

In [ ]:
corr_matrix = strong_corr

plt.figure(figsize=(16, 12))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90,
	fontsize=18
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
	fontsize=18
)

plt.title("Correlation Matrix of Model Features with absolut correlation higher than 0.8")
plt.tight_layout()
plt.show()

In [ ]:
colsToDrop = df_final.columns[
    df_final.columns.str.contains("pers-dat-sprach-mutt")
].tolist()

colsToDrop.append("orgRef_Team Ukraine")

df_caregivers = df_final.drop(columns=colsToDrop)

In [ ]:
colsToCombine = [
    col for col in df_caregivers.columns
    if "PFKomp" in col and col != "PFKomp_mean"
]

colsToCombine


df_caregivers["PFKomp_mean"] = (
    df_caregivers[colsToCombine]
    .mean(axis=1)
    .round()
    .clip(1, 5)
)


df_toExport = df_caregivers.drop(columns=colsToCombine)

In [ ]:
df_toExport.info()

In [ ]:
df_toExport.to_csv("caregivers.csv",sep=";",index=False)